# NODE LIGHTUP ANALYSIS 18NOV2025

# Import Libraries

In [1]:
import h5py
import re
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.models import load_model
import os

2025-12-04 16:25:33.088250: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-04 16:25:33.147203: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-04 16:25:37.752517: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-04 16:25:50.842916: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

# Import the data to be used for the Node Light Up Technique

In [2]:
# Import expression data and transcript data from filtered h5 file
filtered_file_tf = h5py.File("/scratch/by2372/pipeline/ARCHS4/filtered_ARCHS4_TFs.h5", 'r')
ensembl_id = filtered_file_tf['ensembl_id'][...]
symbol = filtered_file_tf['symbol'][...]
expression = filtered_file_tf['data/filtered_expression'][...]
filtered_file_tf.close()

# Read all names and decode bytes to strings
if isinstance(ensembl_id[0], bytes):
    ensembl_id = [s.decode('utf-8') for s in ensembl_id]

# Read all names and decode bytes to strings
if isinstance(symbol[0], bytes):
    symbol = [s.decode('utf-8') for s in symbol]

# Make the transcript ID be the symbol _ ensembl ID
transcript_ID = []
for idx, sym in enumerate(symbol):
    ID = sym + "_" + ensembl_id[idx]
    transcript_ID.append(ID)

In [3]:
expression.shape

(13146, 63324)

In [4]:
len(transcript_ID)

13146

# Implement the Node Light Up technique as a function

In [5]:
def node_light_up(model_name,tf_names,express_mat):
    # load train keras model
    model = load_model('/scratch/by2372/pipeline/ARCHS4/' + model_name + '_trained_NN.keras')
    # create square matrix of zeros. Number of rows and cols is number of TFs
    inp = np.zeros((model.input_shape[1],model.input_shape[1])).T
    # Import cell class names 
    class_names = np.load('/scratch/by2372/pipeline/ARCHS4/' + model_name + '_class_names.npy')
    # Generate baseline TF expression values for the inp square matrix
    tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)
    for i in range(inp.shape[1]):
        inp[i, :] = tf_data.values
    # Generate doubled and halved expression for TFs. Only diagonals of the baseline square matrix is doubled or halved
    half = inp.copy()
    double = inp.copy()
    half[np.eye(half.shape[0]).astype(bool)] = 0.5*half[np.eye(half.shape[0]).astype(bool)]
    double[np.eye(double.shape[0]).astype(bool)] = 2*double[np.eye(double.shape[0]).astype(bool)]

    background_pred = model.predict(inp)
    background_idx = np.argmax(background_pred, axis=1)
    background_labels = class_names[background_idx]
    
    double_pred = model.predict(double)
    double_idx = np.argmax(double_pred, axis=1)
    double_labels = class_names[double_idx]
    
    half_pred = model.predict(half)
    half_idx = np.argmax(half_pred, axis=1)
    half_labels = class_names[half_idx]

    # 7th column is for macrophages classification
    double_background = (double_pred[:,7] - background_pred[:,7]) / background_pred[:,7]
    double_background = pd.DataFrame(double_background)
    double_background.index = tf_names
    double_background.columns = ["Macrophage"]
    double_background_sorted = double_background.sort_values(by = 'Macrophage', ascending=False)
    
    half_background = (half_pred[:,7] - background_pred[:,7]) / background_pred[:,7]
    half_background = pd.DataFrame(half_background)
    half_background.index = tf_names
    half_background.columns = ["Macrophage"]
    half_background_sorted = half_background.sort_values(by = 'Macrophage', ascending=False)

    return double_background_sorted, half_background_sorted

    
    

### Background labels are all macrophage. some double are not macrophage. even less half are not macrophage. I dont think the labels are too useful
### Can i see how changing each TF impacts the change in probability of macrophage the most?

# Feature Selection for Unbalanced

In [6]:
unbalanced_features = node_light_up("unbalanced",transcript_ID,expression)

2025-12-04 16:26:18.177066: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


411/411 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [7]:
unbalanced_features[0].head(n=20) # doubling

,Macrophage
HIF3A_ENST00000475432,0.979409
JUND_ENST00000252818,0.939179
YBX1_ENST00000436427,0.425193
CEBPB_ENST00000303004,0.222598
SON_ENST00000492229,0.194207
ATF4_ENST00000680446,0.185429
NME2_ENST00000503064,0.154106
HIF1A_ENST00000556827,0.132302
SPI1_ENST00000378538,0.127552
ZNF302_ENST00000507959,0.127436


In [8]:
for idx, t in enumerate(unbalanced_features[0].index):
    if "ZNF385A" in str(t):
        print(idx)

609
814
870
874
1200
1508
1785
2900
2925
3025
3253
7485
7767
9582
11416


In [9]:
for idx, t in enumerate(unbalanced_features[0].index):
    if "CEBPA" in str(t):
        print(idx)

12489


In [10]:
for idx, t in enumerate(unbalanced_features[0].index):
    if "CEBPB" in str(t):
        print(idx)

3


In [11]:
unbalanced_features[1].head(n=20) # halving

,Macrophage
NME2_ENST00000512737,0.331719
TERF1_ENST00000518874,0.225042
SOX4_ENST00000244745,0.157592
POU5F1B_ENST00000465342,0.151874
GPBP1L1_ENST00000468724,0.124570
ATF4_ENST00000674920,0.117196
HIF1A_ENST00000337138,0.114611
EGR1_ENST00000239938,0.112011
PA2G4_ENST00000303305,0.107736
LIN28A_ENST00000254231,0.095582


In [12]:
for idx, t in enumerate(unbalanced_features[1].index):
    if "ZNF385A" in str(t):
        print(idx)

1726
3555
5686
5766
9881
10097
10205
10228
11349
11626
11930
12263
12264
12327
12526


In [13]:
for idx, t in enumerate(unbalanced_features[1].index):
    if "CEBPA" in str(t):
        print(idx)

675


In [14]:
for idx, t in enumerate(unbalanced_features[1].index):
    if "CEBPB" in str(t):
        print(idx)

13141


# Feature Selection for Random Oversampler

In [15]:
RO_features = node_light_up("RO",transcript_ID,expression)

411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [16]:
RO_features[0].head(n=20)

,Macrophage
ZNF706_ENST00000517844,0.293495
PIN1_ENST00000247970,0.279903
GTF3A_ENST00000419181,0.199012
GTF3A_ENST00000381140,0.183668
CHCHD3_ENST00000262570,0.160763
AEBP1_ENST00000223357,0.119694
ZNF706_ENST00000519744,0.111481
KLF2_ENST00000592003,0.110017
BATF_ENST00000286639,0.099472
MAF_ENST00000393350,0.091039


In [17]:
for idx, t in enumerate(RO_features[0].index):
    if "ZNF385A" in str(t):
        print(idx)

769
1569
1965
3941
4339
4808
6363
6403
6733
8886
9863
11710
12342
12524
12871


In [18]:
for idx, t in enumerate(RO_features[0].index):
    if "CEBPA" in str(t):
        print(idx)

11603


In [19]:
for idx, t in enumerate(RO_features[0].index):
    if "CEBPB" in str(t):
        print(idx)

13124


In [20]:
RO_features[1].head(n=20)

,Macrophage
SON_ENST00000492229,0.470594
TERF1_ENST00000518874,0.409127
SON_ENST00000475072,0.400367
NME2_ENST00000512737,0.326195
POU5F1B_ENST00000465342,0.296803
ATF4_ENST00000674920,0.182026
HIF3A_ENST00000475432,0.176503
SOX4_ENST00000244745,0.175559
CAMTA1_ENST00000490738,0.174914
ADNP_ENST00000644386,0.173175


In [21]:
for idx, t in enumerate(RO_features[1].index):
    if "ZNF385A" in str(t):
        print(idx)

242
541
706
1371
3167
4143
6524
6637
6706
8183
8815
9070
11163
11539
12418


In [22]:
for idx, t in enumerate(RO_features[1].index):
    if "CEBPA" in str(t):
        print(idx)

11015


In [23]:
for idx, t in enumerate(RO_features[1].index):
    if "CEBPB" in str(t):
        print(idx)

24


# Feature Selection for SMOTE

In [24]:
SMOTE_features = node_light_up("SMOTE",transcript_ID,expression)

411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [25]:
SMOTE_features[0].head(n=20)

,Macrophage
JUNB_ENST00000302754,3.815921
YBX1_ENST00000436427,2.044073
SON_ENST00000492229,1.160269
SON_ENST00000475072,1.088619
ZNF207_ENST00000394673,0.759707
ATF4_ENST00000674920,0.665066
PPARG_ENST00000683749,0.605677
SON_ENST00000300278,0.502542
MAFF_ENST00000624676,0.481438
STAT4_ENST00000392320,0.467749


In [26]:
for idx, t in enumerate(SMOTE_features[0].index):
    if "ZNF385A" in str(t):
        print(idx)

40
392
1570
1766
3016
3184
3230
5170
5547
8780
9993
10735
11249
12432
12942


In [27]:
for idx, t in enumerate(SMOTE_features[0].index):
    if "CEBPA" in str(t):
        print(idx)

13115


In [28]:
for idx, t in enumerate(SMOTE_features[0].index):
    if "CEBPB" in str(t):
        print(idx)

13001


In [29]:
SMOTE_features[1].head(n=20)

,Macrophage
SOX4_ENST00000244745,0.622425
GPBP1L1_ENST00000468724,0.488879
HIF1A_ENST00000337138,0.434042
HMGA1_ENST00000347617,0.349012
SOX2_ENST00000325404,0.314514
HMGN3_ENST00000275036,0.314122
TERF1_ENST00000518874,0.306546
PA2G4_ENST00000552766,0.290012
NME2_ENST00000512737,0.284281
POU5F1B_ENST00000465342,0.279208


In [30]:
for idx, t in enumerate(SMOTE_features[1].index):
    if "ZNF385A" in str(t):
        print(idx)

204
716
1884
2402
3139
4353
7603
7999
9912
9951
10110
11383
11566
12739
13106


In [31]:
for idx, t in enumerate(SMOTE_features[1].index):
    if "CEBPA" in str(t):
        print(idx)

33


In [32]:
for idx, t in enumerate(SMOTE_features[1].index):
    if "CEBPB" in str(t):
        print(idx)

146


# Feature Selection for BLSMOTE

In [33]:
BLSMOTE_features = node_light_up("BLSMOTE",transcript_ID,expression)

411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [34]:
BLSMOTE_features[0].head(n=20)

,Macrophage
PPARG_ENST00000683749,0.112710
HIF3A_ENST00000475432,0.110977
HMGN3_ENST00000620514,0.101328
PA2G4_ENST00000552766,0.096447
KLF6_ENST00000497571,0.092739
SON_ENST00000492229,0.088372
MAFB_ENST00000373313,0.083260
CEBPZ_ENST00000234170,0.077627
CAMTA1_ENST00000473578,0.074894
HES1_ENST00000476918,0.071573


In [35]:
for idx, t in enumerate(BLSMOTE_features[0].index):
    if "ZNF385A" in str(t):
        print(idx)

779
891
1175
1294
1607
2870
3138
6330
9410
9570
10496
10500
10502
12456
12717


In [36]:
for idx, t in enumerate(BLSMOTE_features[0].index):
    if "CEBPA" in str(t):
        print(idx)

1360


In [37]:
for idx, t in enumerate(BLSMOTE_features[0].index):
    if "CEBPB" in str(t):
        print(idx)

218


In [38]:
BLSMOTE_features[1].head(n=20)

,Macrophage
TERF1_ENST00000518874,0.155333
SON_ENST00000475072,0.128511
POU5F1B_ENST00000465342,0.124887
FOS_ENST00000303562,0.094235
HMGA1_ENST00000311487,0.049598
STAT6_ENST00000300134,0.046327
STAT1_ENST00000361099,0.043021
YBX1_ENST00000321358,0.042099
HMGA1_ENST00000347617,0.041525
SON_ENST00000492229,0.038173


In [39]:
for idx, t in enumerate(BLSMOTE_features[1].index):
    if "ZNF385A" in str(t):
        print(idx)

429
695
2632
2641
2642
3566
3722
6810
10000
10266
11546
11850
11967
12250
12357


In [40]:
for idx, t in enumerate(BLSMOTE_features[1].index):
    if "CEBPA" in str(t):
        print(idx)

1079


In [41]:
for idx, t in enumerate(BLSMOTE_features[1].index):
    if "CEBPB" in str(t):
        print(idx)

12844
